# 📈 Forecasting com TEMPO

---

**Ficha Técnica do Modelo**

| Campo | Valor |
|-------|-------|
| **Modelo** | TEMPO — Prompt-based GPT para séries temporais (zero-shot) |
| **Biblioteca** | `tempo` (ambiente `venv_tempo`) — `TEMPO` |
| **Versão do modelo** | `Melady/TEMPO` / `TEMPO-80M_v1.pth` (80M parâmetros) |
| **Hiperparâmetros configurados** | `context_length=96`, `seq_len=336` (fixo pela arquitetura), `SEED=42` |
| **Busca de hiperparâmetros** | Não (zero-shot — sem fine-tuning) |
| **Critério de seleção** | N/A |
| **Séries utilizadas** | 29 séries com total ≥ 36 observações (`len(train_series) < TEST_SIZE + 12`) |
| **Horizonte** | 3 meses (`HORIZON = 3`) |
| **Protocolo de avaliação** | Walk-forward expansível, 24 meses de teste (`TEST_SIZE = 24`), janelas de 3 meses; inferência via `model.forward()` direto (contorno do bug de dupla normalização RevIN) |
| **Reprodutibilidade** | `SEED = 42` (`random.seed` + `np.random.seed` + `torch.manual_seed` + `cuda.manual_seed_all`) |
| **Referência** | Cao, D. et al. (2024). TEMPO: Prompt-based Generative Pre-trained Transformer for Time Series Forecasting. *ICLR 2024*. DC-Research. |

---

In [ ]:
# ── Semente global para reprodutibilidade ──
import random, numpy as np, torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"🔒 Seed fixada: {SEED} (random + numpy + torch)")

## 1. Importação de Bibliotecas

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Importar TEMPO
from tempo.models.TEMPO import TEMPO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print("✅ Bibliotecas carregadas!")

c:\Users\phill\Projetos\lag-llama\notebooks\venv_tempo\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 1.13.0+cpu
CUDA available: False
✅ Bibliotecas carregadas!


## 2. Carregamento dos Dados

In [2]:
# Carregar base econômica
df = pd.read_csv('base_economica_brasil.csv', index_col=0, parse_dates=True)

# Lista de todas as séries
ALL_SERIES = [col for col in df.columns]
# Excluir PIM e IPCA_mensal da análise
ALL_SERIES = [s for s in ALL_SERIES if s not in ['PIM', 'IPCA_mensal']]

print(f"📊 Base carregada: {len(df)} observações")
print(f"📈 Séries disponíveis: {len(ALL_SERIES)}")
print(f"📅 Período: {df.index.min().strftime('%Y-%m')} a {df.index.max().strftime('%Y-%m')}")
print(f"\n🔢 Séries: {ALL_SERIES}")
df.head()

📊 Base carregada: 108 observações
📈 Séries disponíveis: 35
📅 Período: 2017-01 a 2025-12

🔢 Séries: ['IBC_Br', 'Selic', 'Cambio_USDBRL', 'Desemprego', 'Brent_USD', 'Soja_USD', 'Minerio_USD', 'Ibovespa', 'ICC_FGV', 'Credito_Total', 'Inadimplencia', 'Massa_Salarial', 'CPI_USA', 'Prod_Ind_USA', 'Cafe_USD', 'Ouro_USD', 'GasNatural_USD', 'Cobre_USD', 'ETF_Emergentes', 'IGP_DI', 'INCC', 'ICE_Empresarial', 'Housing_Starts_EUA', 'Dollar_Index_Fed', 'PMS_Volume', 'PMC_Ampliado', 'IGPM', 'INPC', 'M2', 'Divida_PIB', 'Vendas_Varejo', 'Balanca_Comercial', 'NUCI_FGV', 'EAI_Emprego_Ind', 'SP500']


,IBC_Br,Selic,Cambio_USDBRL,Desemprego,Brent_USD,Soja_USD,Minerio_USD,Ibovespa,ICC_FGV,Credito_Total,...,PMC_Ampliado,IGPM,INPC,M2,Divida_PIB,Vendas_Varejo,Balanca_Comercial,NUCI_FGV,EAI_Emprego_Ind,SP500
2017-01-01,90.56860,13.17,3.1270,12.7,55.25,379.589979,80.818182,64671.0,102.25,1537976.0,...,-0.1,0.64,0.42,5.842420e+09,46.46,89.14,2427.0,73.2,514650.2,2278.870117
2017-02-01,90.92437,12.82,3.0993,13.3,53.36,380.872624,88.950000,66662.0,113.80,1535492.0,...,-4.8,0.08,0.24,5.861693e+09,47.26,82.01,4368.2,73.7,1024989.8,2363.639893
2017-03-01,99.65199,12.15,3.1684,13.9,52.20,366.095056,87.195652,64984.0,109.38,1540450.0,...,-1.9,0.01,0.32,5.936526e+09,47.53,88.52,6418.6,73.3,1585673.4,2362.719971
2017-04-01,93.78125,11.59,3.1984,13.7,49.46,347.861310,70.400000,65403.0,109.01,1530470.0,...,-0.5,-1.10,0.08,5.925396e+09,47.48,88.31,6125.7,73.4,2123135.2,2384.199951
2017-05-01,95.21290,11.15,3.2437,13.4,49.40,350.179987,61.630435,62711.0,103.49,1526937.0,...,4.9,-0.93,0.36,5.947256e+09,48.01,90.43,6712.1,74.0,2673694.8,2411.800049


In [3]:
# Configurar device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Device: {device}")

# Carregar modelo TEMPO pré-treinado do HuggingFace
print("\n⏳ Carregando modelo TEMPO-80M do HuggingFace...")
print("   (Isso pode demorar na primeira vez - download ~300MB)")

model = TEMPO.load_pretrained_model(
    device=device,
    repo_id="Melady/TEMPO",
    filename="TEMPO-80M_v1.pth",
    cache_dir="./checkpoints/TEMPO_checkpoints"
)

print("\n✅ Modelo TEMPO carregado com sucesso!")

🖥️ Device: cpu

⏳ Carregando modelo TEMPO-80M do HuggingFace...
   (Isso pode demorar na primeira vez - download ~300MB)
------------------No need to load pretrained GPT model------------------
trainable params: 294912 || all params: 82207488
Loading model from: ./checkpoints/TEMPO_checkpoints/TEMPO-80M_v1_checkpoint.pth

✅ Modelo TEMPO carregado com sucesso!


## 3. Configuração do Experimento

In [4]:
# Parâmetros de previsão (PADRONIZADO)
HORIZON = 3              # 3 meses à frente
TEST_SIZE = 24           # últimos 24 meses para teste
context_length = 96      # Contexto para TEMPO

print(f"⚙️ Configuração:")
print(f"   - Horizonte de previsão: {HORIZON} meses")
print(f"   - Tamanho do teste: {TEST_SIZE} meses")
print(f"   - Contexto: {context_length} pontos")
print(f"   - Modelo: TEMPO-80M (Zero-Shot)")

⚙️ Configuração:
   - Horizonte de previsão: 3 meses
   - Tamanho do teste: 12 meses
   - Contexto: 96 pontos
   - Modelo: TEMPO-80M (Zero-Shot)


In [ ]:
def calcular_metricas(real, previsto):
    """Calcula MAE, RMSE e MAPE."""
    real = np.array(real)
    previsto = np.array(previsto)
    
    mae = np.mean(np.abs(real - previsto))
    rmse = np.sqrt(np.mean((real - previsto) ** 2))
    
    # MAPE com epsilon para evitar divisão por zero
    mape = np.mean(np.abs((real - previsto) / (real + 1e-8))) * 100
    
    return mae, rmse, mape


def forecast_with_tempo(train_series, test_series, model, horizon=HORIZON, ctx_len=96):
    """
    Realiza previsão com TEMPO usando walk-forward validation.
    
    CORREÇÃO: Usa model.forward() diretamente em vez de model.predict().
    O método predict() tem um bug de dupla normalização RevIn:
      1. predict() normaliza a entrada com rev_in_trend('norm')
      2. forward() normaliza NOVAMENTE, sobrescrevendo mean/stdev
      3. forward() desnormaliza com os valores errados → saída em escala normalizada
    
    A solução é chamar forward() diretamente, que faz norm→process→denorm corretamente.
    """
    seq_len = model.seq_len  # Comprimento esperado pelo modelo (336)
    all_preds = []
    all_actuals = []
    
    # Walk-forward no período de teste
    for i in range(0, len(test_series), horizon):
        # Dados de treino até o momento atual
        if i > 0:
            current_train = pd.concat([train_series, test_series.iloc[:i]])
        else:
            current_train = train_series.copy()
        
        # Preencher NaN
        current_train = current_train.ffill().bfill()

        try:
            # Preparar contexto (últimos ctx_len pontos)
            context = current_train.values[-ctx_len:] if len(current_train) >= ctx_len else current_train.values
            
            # Número de passos a prever
            n_steps = min(horizon, len(test_series) - i)
            
            # Preparar tensor na forma [1, L, 1]
            x = torch.FloatTensor(context).unsqueeze(0).unsqueeze(2).to(next(model.parameters()).device)
            B, L, M = x.shape
            
            # Padding para seq_len (mesma lógica do predict, mas SEM normalizar antes)
            if L > seq_len:
                x = x[:, -seq_len:, :]
            elif L < seq_len:
                pad_length = seq_len - L
                if pad_length <= L:
                    x_padded = torch.cat([x] * (seq_len // L + 1), dim=1)[:, :seq_len, :]
                else:
                    padding = torch.zeros(B, pad_length, M, device=x.device)
                    x_padded = torch.cat([padding, x], dim=1)
                x = x_padded
            
            # Chamar forward() diretamente — faz norm+denorm corretos internamente
            # forward() retorna shape [B, pred_len, M] = [1, 96, 1]
            # Precisamos dos PRIMEIROS n_steps passos (t+1..t+3), não dos últimos!
            with torch.no_grad():
                outputs, _ = model.forward(x, test=True)
            
            preds = outputs.cpu().squeeze().numpy()[:n_steps]
            actuals = test_series.iloc[i:i+n_steps].values
            
            # Verificar NaN nas previsões
            if np.any(np.isnan(preds)):
                preds = np.nan_to_num(preds, nan=np.nanmean(current_train.values))
            
            all_preds.extend(preds)
            all_actuals.extend(actuals)
                
        except Exception as e:
            # Fallback: usar média dos últimos valores
            print(f"   Fallback para serie (erro: {str(e)[:50]})")
            n_steps = min(horizon, len(test_series) - i)
            fallback_pred = current_train.iloc[-12:].mean()
            all_preds.extend([fallback_pred] * n_steps)
            all_actuals.extend(test_series.iloc[i:i+n_steps].values)
    
    return np.array(all_preds), np.array(all_actuals)

print("✅ Funções definidas (CORRIGIDO: usa forward() para evitar dupla normalização RevIn)")

✅ Funções definidas (CORRIGIDO: usa forward() para evitar dupla normalização RevIn)


## 4. Treinamento e Previsão (Walk-Forward)

In [6]:
# Dicionário para armazenar resultados
all_results = {}

# Split global treino/teste (padrão de todos os notebooks da dissertação)
train_df = df.iloc[:-TEST_SIZE]
test_df = df.iloc[-TEST_SIZE:]

print("="*80)
print("🚀 INICIANDO PREVISÕES COM TEMPO")
print(f"   Train: {train_df.index[0].strftime('%Y-%m')} a {train_df.index[-1].strftime('%Y-%m')} ({len(train_df)} obs)")
print(f"   Test:  {test_df.index[0].strftime('%Y-%m')} a {test_df.index[-1].strftime('%Y-%m')} ({len(test_df)} obs)")
print("="*80)

for series_name in tqdm(ALL_SERIES, desc="Processando séries"):
    try:
        train_series = train_df[series_name].dropna()
        test_series = test_df[series_name].dropna()
        
        if len(train_series) < TEST_SIZE + 12:
            print(f"⚠️ {series_name}: Poucos dados ({len(train_series)} obs)")
            continue
        
        if len(test_series) == 0:
            print(f"⚠️ {series_name}: Sem dados de teste")
            continue
        
        # Realizar previsão
        preds, actuals = forecast_with_tempo(train_series, test_series, model, horizon=HORIZON)
        
        if len(preds) > 0:
            # Calcular métricas
            mae, rmse, mape = calcular_metricas(actuals, preds)
            
            all_results[series_name] = {
                'mae': mae,
                'rmse': rmse,
                'mape': mape,
                'n_points': len(preds),
                'preds': preds,
                'actuals': actuals,
                'dates': list(test_series.index[:len(preds)])
            }
            print(f"✅ {series_name}: MAE={mae:.2f}, RMSE={rmse:.2f}, MAPE={mape:.2f}%")
        else:
            print(f"❌ {series_name}: Sem resultados válidos")
            
    except Exception as e:
        print(f"❌ {series_name}: Erro - {str(e)[:50]}")

print("\n" + "="*80)
print(f"✅ CONCLUÍDO: {len(all_results)}/{len(ALL_SERIES)} séries processadas")
print("="*80)

🚀 INICIANDO PREVISÕES COM TEMPO
   Train: 2017-01 a 2024-12 (96 obs)
   Test:  2025-01 a 2025-12 (12 obs)


Processando séries:   3%|▎         | 1/35 [01:05<36:51, 65.04s/it]

✅ IBC_Br: MAE=13.05, RMSE=14.21, MAPE=11.90%


Processando séries:   6%|▌         | 2/35 [01:44<27:28, 49.94s/it]

✅ Selic: MAE=3.81, RMSE=4.06, MAPE=26.84%


...

✅ NUCI_FGV: MAE=7.94, RMSE=9.83, MAPE=9.60%


Processando séries:  97%|█████████▋| 34/35 [11:17<00:06,  6.12s/it]

✅ EAI_Emprego_Ind: MAE=4519126.95, RMSE=4849575.50, MAPE=99.29%


Processando séries: 100%|██████████| 35/35 [11:25<00:00, 19.58s/it]

✅ SP500: MAE=1229.48, RMSE=1393.00, MAPE=19.00%

✅ CONCLUÍDO: 35/35 séries processadas


## 5. Resultados e Métricas

In [7]:
# Verificar se há resultados
if len(all_results) == 0:
    print("⚠️ Nenhum resultado válido para exibir!")
else:
    # Criar DataFrame com resultados
    results_df = pd.DataFrame([
        {
            'Série': name,
            'MAE': data['mae'],
            'RMSE': data['rmse'],
            'MAPE (%)': data['mape'],
            'N Pontos': data['n_points']
        }
        for name, data in all_results.items()
    ]).sort_values('MAPE (%)')

    # Adicionar classificação
    def classificar(mape):
        if mape < 10:
            return '⭐ Excelente'
        elif mape < 20:
            return '✅ Muito Bom'
        elif mape < 30:
            return '👍 Bom'
        elif mape < 50:
            return '⚠️ Regular'
        else:
            return '❌ Difícil'

    results_df['Classificação'] = results_df['MAPE (%)'].apply(classificar)
    results_df = results_df.set_index('Série')

    # Exibir tabela
    print("="*80)
    print("📊 RANKING - TEMPO")
    print("="*80)
    print(f"\nModelo: TEMPO-80M | Horizonte: {HORIZON} meses | Teste: {TEST_SIZE} meses\n")
    print(results_df.round(2).to_string())

    # Estatísticas gerais
    print("\n" + "-"*80)
    print("📈 ESTATÍSTICAS GERAIS:")
    print(f"   Total de séries analisadas: {len(results_df)}")
    print(f"   MAE médio geral: {results_df['MAE'].mean():.2f}")
    print(f"   RMSE médio geral: {results_df['RMSE'].mean():.2f}")
    print(f"   MAPE médio geral: {results_df['MAPE (%)'].mean():.2f}%")
    print(f"   Melhor série (MAPE): {results_df['MAPE (%)'].idxmin()} ({results_df['MAPE (%)'].min():.2f}%)")
    print(f"   Pior série (MAPE): {results_df['MAPE (%)'].idxmax()} ({results_df['MAPE (%)'].max():.2f}%)")

📊 RANKING - TEMPO

Modelo: TEMPO-80M | Horizonte: 3 meses | Teste: 12 meses

                             MAE          RMSE      MAPE (%)  N Pontos Classificação
Série                                                                               
Brent_USD           3.430000e+00  4.630000e+00  5.200000e+00        12   ⭐ Excelente
CPI_USA             1.870000e+01  2.674000e+01  5.800000e+00        12   ⭐ Excelente
Cambio_USDBRL       3.500000e-01  4.700000e-01  6.180000e+00        12   ⭐ Excelente
Vendas_Varejo       7.670000e+00  1.058000e+01  6.760000e+00        12   ⭐ Excelente
Divida_PIB          4.530000e+00  5.840000e+00  7.090000e+00        12   ⭐ Excelente
Prod_Ind_USA        7.840000e+00  9.310000e+00  7.740000e+00        12   ⭐ Excelente
Minerio_USD         8.470000e+00  1.218000e+01  8.020000e+00        12   ⭐ Excelente
Dollar_Index_Fed    1.043000e+01  1.342000e+01  8.380000e+00        12   ⭐ Excelente
PMS_Volume          9.680000e+00  1.168000e+01  8.890000e+00        12   

## 6. Visualização: Ranking MAPE por Série

In [ ]:
# ── Gráfico: Ranking MAPE por Série ──
sorted_df = results_df.sort_values('MAPE (%)')

fig, ax = plt.subplots(figsize=(12, 8))

cores = ['#2ecc71' if m < 10 else '#3498db' if m < 20 else '#f39c12' if m < 30 else '#e74c3c'
         for m in sorted_df['MAPE (%)']]

bars = ax.barh(range(len(sorted_df)), sorted_df['MAPE (%)'],
               color=cores, edgecolor='white', height=0.7)
ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels(sorted_df.index)
ax.invert_yaxis()
ax.set_xlabel('MAPE (%)')
ax.set_title(f'TEMPO — MAPE por Série\n(Walk-Forward, h={HORIZON}, teste={TEST_SIZE} meses)',
             fontsize=13, fontweight='bold')
ax.axvline(x=sorted_df['MAPE (%)'].mean(), color='red', linestyle='--',
           label=f'Média: {sorted_df["MAPE (%)"].mean():.1f}%')
ax.legend(loc='lower right')

for i, (bar, val) in enumerate(zip(bars, sorted_df['MAPE (%)'])):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('tempo_mape_por_serie.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: tempo_mape_por_serie.png")

## 7. Visualização: Real vs. Projetado (Top 6 Séries)

In [ ]:
# ── Gráfico: Real vs. Projetado (Top 6 Séries por MAPE) ──
top6 = sorted_df.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, sn in zip(axes.flatten(), top6):
    data = all_results[sn]

    # Valores reais (teste)
    ax.plot(data['dates'], data['actuals'], 'b-o',
            label='Real', markersize=4, linewidth=2)

    # Previsões do modelo
    ax.plot(data['dates'], data['preds'], 'r--s',
            label='Previsão', markersize=4, linewidth=2)

    ax.set_title(f"{sn}\nMAPE: {data['mape']:.1f}%", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45, labelsize=8)

axes.flatten()[0].legend(fontsize=8)
fig.suptitle('TEMPO — Real vs. Projetado (6 Melhores Séries)',
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('tempo_previsoes.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: tempo_previsoes.png")

## 8. Exportação de Resultados

In [10]:
# Salvar resultados com nomes padronizados para compatibilidade com consolidação
if len(all_results) > 0:
    results_export = results_df.reset_index()
    results_export.columns = ['Serie', 'MAE', 'RMSE', 'MAPE', 'N_Pontos', 'Classificacao']
    results_export.to_csv('resultados_tempo.csv', index=False)
    print("💾 Resultados salvos em 'resultados_tempo.csv'")
    print(f"   Colunas: {list(results_export.columns)}")

    # Salvar previsões detalhadas (usando datas armazenadas no walk-forward)
    all_forecasts = []
    for series_name, data in all_results.items():
        for j in range(len(data['preds'])):
            all_forecasts.append({
                'Serie': series_name,
                'Data': str(data['dates'][j])[:10] if j < len(data['dates']) else '',
                'Previsao': data['preds'][j]
            })
    df_prev = pd.DataFrame(all_forecasts)
    df_prev.to_csv('previsoes_tempo.csv', index=False)
    print(f"💾 Previsões salvas em 'previsoes_tempo.csv' ({len(df_prev)} linhas)")

💾 Resultados salvos em 'resultados_tempo.csv'
   Colunas: ['Serie', 'MAE', 'RMSE', 'MAPE', 'N_Pontos', 'Classificacao']
💾 Previsões salvas em 'previsoes_tempo.csv' (420 linhas)


## 📊 Resumo Final

### TEMPO - Foundation Model para Séries Temporais

**Características:**
- Foundation Model baseado em GPT (80M parâmetros)
- Prompt-based generative approach
- Zero-shot forecasting
- Publicado no ICLR 2024

**Paper:** [TEMPO: Prompt-based Generative Pre-trained Transformer for Time Series Forecasting](https://arxiv.org/pdf/2310.04948)

---

### ⚠️ Limitação Estrutural do TEMPO para Séries Mensais Curtas

O TEMPO apresentou MAPE médio significativamente superior ao dos demais Foundation Models
(Chronos, Lag-Llama, TimesFM). A análise do código-fonte revelou que isso decorre de uma
**limitação arquitetural** do modelo, e não de erro de implementação:

#### 1. Incompatibilidade de comprimento de contexto

O TEMPO foi projetado com `seq_len=336` pontos fixos de entrada. Suas camadas lineares
internas (projeção de patches, saída de tendência/sazonalidade/ruído) são **dimensionadas
rigidamente** para esse comprimento. A base utilizada nesta dissertação possui 108 observações
mensais (jan/2017–dez/2025), das quais no máximo 84–96 estão disponíveis como contexto
no walk-forward. Isso representa apenas **25–29% do comprimento esperado**.

#### 2. Padding corrompe o pipeline interno

Para preencher os 336 pontos exigidos, é necessário aplicar padding artificial (zero-padding
ou repetição da série). Esse padding compromete três mecanismos internos do modelo:

| Componente | Efeito do Padding |
|------------|-------------------|
| **RevIN** (Reversible Instance Normalization) | Calcula média e desvio padrão sobre os 336 pontos, incluindo ~240 valores artificiais. A normalização fica distorcida. |
| **Decomposição STL** (trend/season/noise) | As camadas lineares de decomposição foram treinadas em sequências contínuas de 336 pontos; recebem 71% de dados artificiais. |
| **Patches + GPT-2** | Dos ~41 patches tokenizados, ~30 contêm apenas padding. O mecanismo de atenção processa majoritariamente ruído. |

#### 3. Bug na API oficial (`predict()`)

O método `predict()` da biblioteca TEMPO possui um bug de **dupla normalização RevIN**:
a entrada é normalizada em `predict()` e novamente em `forward()`, sobrescrevendo as estatísticas
de média/desvio. A desnormalização subsequente utiliza valores incorretos. Este bug permanece
não corrigido no repositório oficial (último commit: Fev/2025). A solução adotada neste
notebook — chamar `forward()` diretamente — evita a dupla normalização, mas não resolve o
problema do padding.

#### 4. Comparação com outros Foundation Models

| Modelo | Contexto requerido | Séries de 84–108 pontos | Padding necessário? |
|--------|--------------------|-------------------------|---------------------|
| **TEMPO** | 336 (fixo) | 25–29% do requerido | Sim — 71–75% artificial |
| **Chronos** | Variável | Suficiente | Não |
| **Lag-Llama** | 32 | Excede o requerido | Não |
| **TimesFM** | Variável | Suficiente | Não |

#### 5. Conclusão

A performance inferior do TEMPO neste experimento não invalida o modelo em si — que obteve
resultados competitivos em benchmarks com séries longas (ETT, Traffic, Weather) — mas evidencia
que **Foundation Models com comprimento de entrada fixo podem ser inadequados para séries
temporais mensais curtas**, cenário comum em economia e finanças. Esta é uma contribuição
relevante da análise comparativa realizada nesta dissertação.